# Health Inequality Project

This notebook is looking at health inequality within England, by Local Authority. 
Local Authorities (LA) are specific geopgraphic regions such as districts, boroughs, or countys. An example of an LA would be Birmingham.

In this project, I am answering the question: 'how does health and life expectancy vary across Local authoirites depending on demographic factors such as deprivation, education levels?'


This notebook analyses health inequality across the UK using indicators such as life expectancy, disease-related mortality, diagnostic waiting times, and access to care.

These measures are examined across geography, demographics, deprivation, and education to understand where disparities are most pronounced.

Most datasets used are from 2023, with ethnicity and deprivation data sourced from their latest available releases.

Datasets includes:
- Ethnicity
- Life expectancy 2023
- deprivation 2019
- Cause of death registration by La 2023
- Travel time to GPs 
- Travel times to hospitals 
- Vaccination children by area 2023
- Number of GP registrations by La
- QOF datasets
- Pollotions data
- Population Size


In [2]:
%load_ext sql
%config SqlMagic.style = 'PLAIN_COLUMNS'   
%config SqlMagic.autopandas = True         

In [3]:
import pandas as pd
import sqlite3

# 1. This Section is for Importing Datasets

### a) Postcode Lookups 

In [1]:
# Postcode Lookups.
#The latest postcode look ups for SICB, ICB, Postcode and Practices. 

df1= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/ukpostcodesandladaugust2023.csv")
df2= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/postcode_nov_2025_lookup.csv")
df3= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/LSOA_SICBL_ICB_LAD_April_2023_Lookup_EN.csv")
df4= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/epraccur.csv")


conn = sqlite3.connect("health_inequality.db")

df1.to_sql("ukpostcodesandladaugust2023", conn, index=False, if_exists="replace")
df2.to_sql("postcode_nov_2025_lookup", conn, index=False, if_exists="replace")
df3.to_sql("LSOA_SICBL_ICB_LAD_April_2023_Lookup_EN", conn, index=False, if_exists="replace")
df4.to_sql("epraccur", conn, index=False, if_exists="replace")


NameError: name 'pd' is not defined

In [ ]:
query = """
DROP VIEW IF EXISTS epraccur_lookup;

CREATE VIEW epraccur_lookup 
AS
SELECT 
`Organisation Code` AS org_code,
Name AS name,
`National Grouping` AS nat_group,
`High Level Health Geography` AS high_lvl_hlth_geo,	
`Address Line 1` AS address_line_one,
`Address Line 2` AS address_line_two,
`Address Line 3` AS address_line_three,
Postcode AS postcode,
Status AS status,
`Organisation Sub-Type Code` AS org_sub_type_code,
Commissioner AS commissioner,	
`Join Provider/Purchaser Date` AS join_provdr_purch_date,
`Left Provider/Purchaser Date` AS left_provdr_purche_date,
`Contact Telephone Number` AS contact_number,
`Amended Record Indicator`	AS amdd_recd_indicator,
`Provider/Purchaser` AS provider_purchaser,
`Prescribing Setting` AS prscbng_setng
FROM 
    epraccur;
"""

conn.executescript(query)  
conn.commit()


In [ ]:

query = """
DROP VIEW IF EXISTS ukpostcodesandlad;

CREATE VIEW ukpostcodesandlad 
AS
SELECT 
    Postcode as postcode,
    `LAD Code` AS lad_code,
    `LAD name` AS lad_name
FROM 
    ukpostcodesandladaugust2023 ;
"""

conn.executescript(query)  
conn.commit()


### a.5) Population 

Pulling in Estimation of population datsets to create weighted life expectancy, death rates by population, 

In [6]:

dfm= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/est_pop_mal_mdy23.csv")
dff= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/est_pop_fem_mdy23.csv")
dam= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/est_pop_all_mdy23.csv")


conn = sqlite3.connect("health_inequality.db")
dfm.to_sql("est_pop_mal_mdy23", conn, index=False, if_exists="replace")
dff.to_sql("est_pop_fem_mdy23", conn, index=False, if_exists="replace")
dam.to_sql("est_pop_all_mdy23", conn, index=False, if_exists="replace")
dam

,Code,Name,Geography,All ages,0,1,2,3,4,5,...,81,82,83,84,85,86,87,88,89,90+
0,K04000001,ENGLAND AND WALES,Country,"60,854,727","600,801","641,879","638,385","661,083","672,685","681,900",...,"335,360","288,359","287,413","271,872","250,133","222,324","195,657","170,123","142,674","551,758"
1,E92000001,ENGLAND,Country,"57,690,323","573,100","611,983","608,924","629,972","640,658","648,548",...,"314,801","270,597","270,666","256,427","235,873","209,664","184,536","160,616","134,632","521,291"
2,E12000001,NORTH EAST,Region,"2,711,380","24,711","26,256","26,370","26,959","28,242","28,768",...,"15,345","13,718","13,665","12,776","11,874","10,641","8,949","8,003","6,504","23,914"
3,E06000047,County Durham,Unitary Authority,"532,182","4,410","4,736","4,805","4,964","5,084","5,199",...,"3,180","2,922","2,790","2,542","2,361","2,164","1,825","1,596","1,284","4,497"
4,E06000005,Darlington,Unitary Authority,"110,562","1,010","1,115","1,102","1,157","1,168","1,210",...,633,592,613,554,503,448,377,354,277,"1,083"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
352,W06000018,Caerphilly,Unitary Authority,"176,437","1,599","1,713","1,609","1,807","1,848","1,990",...,"1,131",927,872,814,717,634,551,458,392,"1,244"
353,W06000019,Blaenau Gwent,Unitary Authority,"67,356",678,742,655,730,748,742,...,458,351,336,307,264,222,241,171,138,483
354,W06000020,Torfaen,Unitary Authority,"93,419",928,975,934,"1,027","1,037","1,048",...,611,545,461,451,412,341,349,285,252,813
355,W06000021,Monmouthshire,Unitary Authority,"94,572",738,796,830,828,828,878,...,807,657,666,571,524,477,442,379,322,"1,287"


In [45]:
query = """
DROP VIEW IF EXISTS est_pop_mal;
DROP VIEW IF EXISTS est_pop_fem;
DROP VIEW IF EXISTS est_pop_all;

CREATE VIEW est_pop_mal 
AS
SELECT 
    Code AS code,
    Name AS name,
    Geography AS geography,
    CAST(REPLACE(`All ages` , ',', '') AS INTEGER) AS num_mls_all_ages
FROM 
    est_pop_mal_mdy23;

CREATE VIEW est_pop_fem 
AS
SELECT 
    Code AS code,
    Name AS name,
    Geography AS geography,
    CAST(REPLACE(`All ages` , ',', '') AS INTEGER) AS num_fem_all_ages
FROM 
    est_pop_fem_mdy23;
 
CREATE VIEW est_pop_all 
AS
SELECT 
    Code AS code,
    Name AS name,
    Geography AS geography,
    CAST(REPLACE(`All ages` , ',', '') AS INTEGER) AS num_all_ages
FROM 
    est_pop_all_mdy23;
"""

conn.executescript(query)  
conn.commit()

In [46]:
check = pd.read_sql("SELECT * FROM est_pop_mal LIMIT 5;", conn)
print(check.head())


        code               name          geography  num_mls_all_ages
0  K04000001  ENGLAND AND WALES            Country          29835992
1  E92000001            ENGLAND            Country          28283074
2  E12000001         NORTH EAST             Region           1329511
3  E06000047      County Durham  Unitary Authority            259564
4  E06000005         Darlington  Unitary Authority             54068


### b) Life Expectancy

The orginal dataset consists of data from 2001 - 2023, All data prior to 2021 has been removed.

In [33]:


df= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/lifeexpectancylocalareas_period_2001_to_2023.csv")
df.head()

conn = sqlite3.connect("health_inequality.db")
df.to_sql("life_expectancyy", conn, index=False, if_exists="replace")

df 


,Period,Country,Area type,Area code,Area name,Sex,Sex code,Age band,Age group,Life expectancy,Lower confidence interval,Upper confidence interval
0,2001 to 2003,England,Local Areas,E06000001,Hartlepool,Male,1,1,<1,73.42,72.68,74.16
1,2001 to 2003,England,Local Areas,E06000001,Hartlepool,Male,1,2,01 to 04,72.90,72.21,73.58
2,2001 to 2003,England,Local Areas,E06000001,Hartlepool,Male,1,3,05 to 09,68.94,68.26,69.61
3,2001 to 2003,England,Local Areas,E06000001,Hartlepool,Male,1,4,10 to 14,63.97,63.30,64.65
4,2001 to 2003,England,Local Areas,E06000001,Hartlepool,Male,1,5,15 to 19,59.00,58.33,59.67
...,...,...,...,...,...,...,...,...,...,...,...,...
335155,2021 to 2023,Wales,Country,W92000004,Wales,Female,2,16,70 to 74,16.47,16.40,16.54
335156,2021 to 2023,Wales,Country,W92000004,Wales,Female,2,17,75 to 79,12.76,12.69,12.82
335157,2021 to 2023,Wales,Country,W92000004,Wales,Female,2,18,80 to 84,9.41,9.34,9.47
335158,2021 to 2023,Wales,Country,W92000004,Wales,Female,2,19,85 to 89,6.66,6.60,6.72


In [34]:
##Sanity Check  
pd.read_sql("PRAGMA table_info(life_expectancyy);", conn)


,cid,name,type,notnull,dflt_value,pk
0,0,Period,TEXT,0,None,0
1,1,Country,TEXT,0,None,0
2,2,Area type,TEXT,0,None,0
3,3,Area code,TEXT,0,None,0
4,4,Area name,TEXT,0,None,0
5,5,Sex,TEXT,0,None,0
6,6,Sex code,INTEGER,0,None,0
7,7,Age band,INTEGER,0,None,0
8,8,Age group,TEXT,0,None,0
9,9,Life expectancy,REAL,0,None,0


In [35]:

query = """
DROP VIEW IF EXISTS life_expect;

CREATE VIEW life_expect 
AS
SELECT 
    area_types,
    area_code,
    epf.code,
    area_name,
    sex,
    sex_code,
    age_band,
    age_group,
    life_expectancy,
    num_mls_all_ages,
    num_fem_all_ages,
    (SUM(
        CASE 
            WHEN sex = 'Male' THEN life_expectancy * num_mls_all_ages
            WHEN sex = 'Female' THEN life_expectancy * num_fem_all_ages 
        END)) 
    / (MAX(num_fem_all_ages) + MAX(num_mls_all_ages)) weighted_le
FROM
( 
SELECT 
    Period as period,
    substr(Period, 1, 4) AS start_year,
    substr(Period, 9, 12)AS end_year,
    Country as country,
    `Area type` as area_types,
    `Area code` as area_code,
    `Area name` as area_name,
     Sex as sex,
    `Sex code` as sex_code,
    `Age band` as age_band,
    `Age group` as age_group,
    `Life expectancy` as life_expectancy,
    `Lower confidence interval` as lower_confidence_interval,
    `Upper confidence interval` as upper_confidence_interval
FROM 
    life_expectancyy lex
WHERE 
    start_year = '2021' 
    AND end_year = '2023'
    AND `Area type` = 'Local Areas'
    AND Country = 'England' 
    AND age_group = '<1' ) le
INNER JOIN 
    est_pop_mal epm
ON
    epm.code = le.area_code
INNER JOIN
    est_pop_fem epf
ON 
    epf.code = le.area_code
GROUP BY 
    le.area_code, 
    area_name;
"""
query = """
SELECT 
    *
FROM 
    life_expect
 """
 
result_df = pd.read_sql(query, conn)
result_df





,area_types,area_code,code,area_name,sex,sex_code,age_band,age_group,life_expectancy,num_mls_all_ages,num_fem_all_ages,weighted_le
0,Local Areas,E06000001,E06000001,Hartlepool,Male,1,1,<1,76.66,46433,48933,78.327599
1,Local Areas,E06000002,E06000002,Middlesbrough,Male,1,1,<1,75.52,76395,76255,77.702996
2,Local Areas,E06000003,E06000003,Redcar and Cleveland,Male,1,1,<1,77.01,67012,70926,79.113027
3,Local Areas,E06000004,E06000004,Stockton-on-Tees,Male,1,1,<1,78.17,99829,102586,79.933700
4,Local Areas,E06000005,E06000005,Darlington,Male,1,1,<1,77.62,54068,56494,79.561691
...,...,...,...,...,...,...,...,...,...,...,...,...
289,Local Areas,E09000029,E09000029,Sutton,Male,1,1,<1,80.44,102299,108824,82.285322
290,Local Areas,E09000030,E09000030,Tower Hamlets,Male,1,1,<1,78.01,164959,163667,79.952334
291,Local Areas,E09000031,E09000031,Waltham Forest,Male,1,1,<1,79.12,134676,141304,81.342115
292,Local Areas,E09000032,E09000032,Wandsworth,Male,1,1,<1,80.34,156984,174472,82.556062


### c) GP Surgeries 
#### i. Number of GP Surgeries in Local Authority Districts

In [ ]:
df= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/numofgpsurgeries_eng_and_wales_ 2024.csv")
df.head()

conn = sqlite3.connect("health_inequality.db")
df.to_sql("num_gp_surgeries", conn, index=False, if_exists="replace")
df 

,LAD code,LAD name,Rural Urban Classification (Note 1),Count (Note 3),"GPs per 100,000 people"
0,E06000001,Hartlepool,Urban with City and Town,16,16.78
1,E06000002,Middlesbrough,Urban with City and Town,22,14.41
2,E06000003,Redcar and Cleveland,Urban with Significant Rural,18,13.05
3,E06000004,Stockton-on-Tees,Urban with City and Town,23,11.36
4,E06000005,Darlington,Urban with City and Town,12,10.85
...,...,...,...,...,...
313,W06000020,Torfaen,Wales,11,11.77
314,W06000021,Monmouthshire,Wales,11,11.63
315,W06000022,Newport,Wales,16,9.78
316,W06000023,Powys,Wales,17,12.65


In [24]:
##Sanity Check
pd.read_sql("PRAGMA table_info(num_gp_surgeries);", conn)


,cid,name,type,notnull,dflt_value,pk
0,0,LAD code,TEXT,0,None,0
1,1,LAD name,TEXT,0,None,0
2,2,Rural Urban Classification (Note 1),TEXT,0,None,0
3,3,Count (Note 3),TEXT,0,None,0
4,4,"GPs per 100,000 people",TEXT,0,None,0


In [31]:

query = """
DROP VIEW IF EXISTS num_gpsurgeries;

CREATE VIEW num_gpsurgeries 
AS
SELECT 
    `LAD code` AS lad_code,
    `LAD name` AS lad_name,
    `Rural Urban Classification (Note 1)` AS rural_urban_classification,
    CAST(REPLACE(`Count (Note 3)`,',','') AS INTEGER) AS count,
    CAST(`GPs per 100,000 people` AS REAL) AS gp_per_hundreadthousand 
FROM 
    num_gp_surgeries
WHERE 
`Rural Urban Classification (Note 1)` != 'Wales';
"""
conn.executescript(query)

query = """
SELECT 
    *
FROM 
    num_gpsurgeries
ORDER BY 
    lad_code
 """
 
result_df = pd.read_sql(query, conn)
result_df


,lad_code,lad_name,rural_urban_classification,count,gp_per_hundreadthousand
0,E06000001,Hartlepool,Urban with City and Town,16,16.78
1,E06000002,Middlesbrough,Urban with City and Town,22,14.41
2,E06000003,Redcar and Cleveland,Urban with Significant Rural,18,13.05
3,E06000004,Stockton-on-Tees,Urban with City and Town,23,11.36
4,E06000005,Darlington,Urban with City and Town,12,10.85
...,...,...,...,...,...
291,E09000029,Sutton,Urban with Major Conurbation,22,10.42
292,E09000030,Tower Hamlets,Urban with Major Conurbation,33,10.04
293,E09000031,Waltham Forest,Urban with Major Conurbation,39,14.13
294,E09000032,Wandsworth,Urban with Major Conurbation,47,14.18


#### ii) Patients registered at a GP Practice

In [ ]:

df= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/gp_reg_pat_prac_all_dec_2023.csv")


conn = sqlite3.connect("health_inequality.db")
df.to_sql("gp_reg_pat_prac_2023", conn, index=False, if_exists="replace")


6328

In [42]:

query = """
DROP VIEW IF EXISTS gp_reg_pat_prac;

CREATE VIEW gp_reg_pat_prac 
AS
SELECT 
    grp.code,
    SUM(grp.num_of_patients_registered_gps) sum_patients_reg_la,
    grp.postcode,
    pcd.pcds,
    pcd.ladnm,
    pcd.ladcd
FROM 
    (
    SELECT 
    PUBLICATION AS publication,
    EXTRACT_DATE AS extract_date,
    TYPE AS type,
    SUB_ICB_LOCATION_CODE AS sub_icb_location_code,
    ONS_SUB_ICB_LOCATION_CODE AS ons_sub_icb_location_code,
    CODE AS code,
    POSTCODE AS postcode,
    SEX AS sex,
    AGE AS age,
    NUMBER_OF_PATIENTS AS num_of_patients_registered_gps
FROM 
    gp_reg_pat_prac_2023   
        
    ) grp
LEFT JOIN 
    postcode_nov_2025_lookup pcd
ON 
    grp.postcode = pcd.pcds
WHERE 
    pcd.ladnm IS NOT NULL
GROUP BY 
    pcd.ladnm,
    pcd.ladnm
"""

conn.executescript(query)  

query = """
SELECT 
    *
FROM 
    gp_reg_pat_prac
 """

result_df = pd.read_sql(query, conn)
result_df


,code,sum_patients_reg_la,postcode,pcds,ladnm,ladcd
0,H82091,65310,BN15 8AN,BN15 8AN,Adur,E07000223
1,C81094,142605,DE4 5PB,DE4 5PB,Amber Valley,E07000032
2,H82087,171625,BN12 5HJ,BN12 5HJ,Arun,E07000224
3,C84053,131180,NG15 6DY,NG15 6DY,Ashfield,E07000170
4,G82730,140394,TN23 3ED,TN23 3ED,Ashford,E07000105
...,...,...,...,...,...,...
291,H82041,119454,BN11 1XE,BN11 1XE,Worthing,E07000229
292,M81007,125864,GL20 7QN,GL20 7QN,Wychavon,E07000238
293,P81079,110654,FY5 2TZ,FY5 2TZ,Wyre,E07000128
294,M81010,110293,DY10 2BG,DY10 2BG,Wyre Forest,E07000239


### d) Deprivation Score

This is the latest deprivation data. As there is not LSOA by population, the simple average is being found rather the weighting.

In [44]:
df= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/imd_2019_index_of_multiple_deprivation.csv")
dff= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/domains_of_deprivation.csv")

 
conn = sqlite3.connect("health_inequality.db")
df.to_sql("num_deprivationscore", conn, index=False, if_exists="replace")
dff.to_sql("domains_of_deprivation", conn, index=False, if_exists="replace")


df

,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Rank,Index of Multiple Deprivation (IMD) Decile
0,E01000001,City of London 001A,E09000001,City of London,"29,199",9
1,E01000002,City of London 001B,E09000001,City of London,"30,379",10
2,E01000003,City of London 001C,E09000001,City of London,"14,915",5
3,E01000005,City of London 001E,E09000001,City of London,"8,678",3
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,"14,486",5
...,...,...,...,...,...,...
32839,E01033764,Liverpool 022E,E08000012,Liverpool,116,1
32840,E01033765,Liverpool 061D,E08000012,Liverpool,945,1
32841,E01033766,Liverpool 042G,E08000012,Liverpool,"12,842",4
32842,E01033767,Liverpool 050J,E08000012,Liverpool,422,1


In [41]:

query = """
DROP VIEW IF EXISTS deprivation_la; 


CREATE VIEW deprivation_la AS
SELECT
    dep.`Local Authority District code (2019)` AS la_code,
    dep.`Local Authority District name (2019)` AS la_name,
    COUNT(*) AS num_lsoas,
    AVG(dep.`Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)`)AS imd_rank_avg,
    AVG(dep.`Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)`)AS imd_decile_avg,
    AVG(dep.`Education, Skills and Training Rank (where 1 is most deprived)`)AS edu_rank_avg,
    AVG(dep.`Health Deprivation and Disability Rank (where 1 is most deprived)`)AS health_rank_avg,
    AVG(dep.`Barriers to Housing and Services Rank (where 1 is most deprived)`)AS barriers_rank_avg,
    AVG(dep.`Living Environment Rank (where 1 is most deprived)`) AS living_env_rank_avg,
    AVG(dep.`Employment Rank (where 1 is most deprived)`) AS employment_rank_avg
FROM domains_of_deprivation dep
GROUP BY
    dep.`Local Authority District code (2019)`,
    dep.`Local Authority District name (2019)`;
"""

conn.executescript(query)  

query = """
SELECT 
    *
FROM 
    deprivation_la

 """
 
result_df = pd.read_sql(query, conn)
result_df


,la_code,la_name,num_lsoas,imd_rank_avg,imd_decile_avg,edu_rank_avg,health_rank_avg,barriers_rank_avg,living_env_rank_avg,employment_rank_avg
0,E06000001,Hartlepool,58,107.603448,3.689655,50.827586,58.551724,23.413793,26.603448,130.051724
1,E06000002,Middlesbrough,86,152.046512,3.383721,116.034884,146.802326,22.965116,22.279070,114.267442
2,E06000003,Redcar and Cleveland,88,60.659091,4.340909,27.988636,80.852273,22.829545,26.556818,74.329545
3,E06000004,Stockton-on-Tees,120,47.575000,5.208333,37.033333,92.600000,21.425000,28.133333,47.400000
4,E06000005,Darlington,65,58.446154,4.815385,45.876923,65.953846,36.861538,24.800000,79.061538
...,...,...,...,...,...,...,...,...,...,...
312,E09000029,Sutton,121,21.132231,7.099174,21.132231,22.859504,14.016529,14.661157,20.644628
313,E09000030,Tower Hamlets,144,9.854167,3.680556,16.944444,12.819444,6.659722,11.111111,13.506944
314,E09000031,Waltham Forest,144,11.222222,4.076389,16.152778,17.993056,45.798611,11.423611,14.638889
315,E09000032,Wandsworth,179,18.212291,6.212291,25.743017,20.821229,28.106145,11.016760,21.229050


In [49]:

query = """
DROP VIEW IF EXISTS deprivation_score;

CREATE VIEW deprivation_score
AS
SELECT 
    `Local Authority District code (2019)` AS la_code,
    `Local Authority District name (2019)` AS la_name,
    COUNT(*) AS num_lsoas,
    AVG(CAST(REPLACE(`Index of Multiple Deprivation (IMD) Rank`, ',', '') AS INT)) AS imd_rank,
    AVG(`Index of Multiple Deprivation (IMD) Decile`) AS imd_decile
FROM 
    num_deprivationscore
GROUP BY 
    `Local Authority District code (2019)`,
    `Local Authority District name (2019)`;
    
"""

conn.executescript(query)  

query = """
SELECT 
    *
FROM 
    deprivation_score

 """
 
result_df = pd.read_sql(query, conn)
result_df


,la_code,la_name,num_lsoas,imd_rank,imd_decile
0,E06000001,Hartlepool,58,10395.965517,3.689655
1,E06000002,Middlesbrough,86,9189.569767,3.383721
2,E06000003,Redcar and Cleveland,88,12470.977273,4.340909
3,E06000004,Stockton-on-Tees,120,15477.158333,5.208333
4,E06000005,Darlington,65,14055.938462,4.815385
...,...,...,...,...,...
312,E09000029,Sutton,121,21630.719008,7.099174
313,E09000030,Tower Hamlets,144,10369.312500,3.680556
314,E09000031,Waltham Forest,144,11735.375000,4.076389
315,E09000032,Wandsworth,179,18713.156425,6.212291


### e) Ethnicity 

The raw data contains 20 enthnic groups at LA level. 
To simplify analysis, categories are grouped into major ethnic groups and aggregated. 

In [ ]:
df= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/census_2021_ethnicity_lad.csv")


conn = sqlite3.connect("health_inequality.db")
df.to_sql("census_enthnicity_2019", conn, index=False, if_exists="replace")

df 

,Lower tier local authorities Code,Lower tier local authorities,Ethnic group (20 categories) Code,Ethnic group (20 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0
1,E06000001,Hartlepool,1,"Asian, Asian British or Asian Welsh: Bangladeshi",278
2,E06000001,Hartlepool,2,"Asian, Asian British or Asian Welsh: Chinese",217
3,E06000001,Hartlepool,3,"Asian, Asian British or Asian Welsh: Indian",335
4,E06000001,Hartlepool,4,"Asian, Asian British or Asian Welsh: Pakistani",297
...,...,...,...,...,...
6615,W06000024,Merthyr Tydfil,15,White: Gypsy or Irish Traveller,56
6616,W06000024,Merthyr Tydfil,16,White: Roma,26
6617,W06000024,Merthyr Tydfil,17,White: Other White,2029
6618,W06000024,Merthyr Tydfil,18,Other ethnic group: Arab,17


In [50]:

query = """
DROP VIEW IF EXISTS census_enthnicity;

CREATE VIEW census_enthnicity
AS
SELECT 
    la_code,
    la_name,
    CASE
        WHEN INSTR(ethnic_group, ':') > 0
        THEN TRIM(SUBSTR(ethnic_group, 1, INSTR(ethnic_group, ':') - 1))
        ELSE ethnic_group
    END AS major_ethnic_group,
    SUM(observation) AS population 
FROM
(  
SELECT 
    `Lower tier local authorities Code` AS la_code,
    `Lower tier local authorities` AS la_name,
    `Ethnic group (20 categories) Code` AS ethnic_group_code,
    `Ethnic group (20 categories)` AS ethnic_group,
    `Observation` AS observation
FROM 
    census_enthnicity_2019) cen
WHERE 
    cen.ethnic_group <> '-8' 
GROUP BY 
    la_code,
    la_name,
    major_ethnic_group;
"""

conn.executescript(query)  
conn.commit()

query = """
SELECT 
    *
FROM 
    census_enthnicity


 """
 
result_df = pd.read_sql(query, conn)
result_df["major_ethnic_group"] = result_df["major_ethnic_group"].replace(
    "", "Other"
)

pivot_df = result_df.pivot_table(
    
    index=["la_code", "la_name"],
    columns="major_ethnic_group",
    values="population",
    aggfunc="sum"
    
).reset_index()

pivot_df = pivot_df.reindex(sorted(pivot_df.columns), axis=1)
pivot_df = pivot_df.rename(columns={
    "Asian, Asian British or Asian Welsh": "asian_pop",
    "Black, Black British, Black Welsh, Caribbean or African": "black_pop",
    "White": "white_pop",
    "Mixed or Multiple ethnic groups": "mixed_pop",
    "Other ethnic group": "other_pop"
})
pivot_df




major_ethnic_group,asian_pop,black_pop,Does not apply,mixed_pop,other_pop,white_pop,la_code,la_name
0,1600,445,0,671,554,89068,E06000001,Hartlepool
1,15090,3816,0,3001,3468,118547,E06000002,Middlesbrough
2,1160,265,0,1185,532,133388,E06000003,Redcar and Cleveland
3,9052,2203,0,2737,1664,180937,E06000004,Stockton-on-Tees
4,2966,701,0,1472,932,101725,E06000005,Darlington
...,...,...,...,...,...,...,...,...
326,1202,228,0,1005,245,89596,W06000020,Torfaen
327,1185,230,0,1115,324,90106,W06000021,Monmouthshire
328,12194,3737,0,4451,2737,136473,W06000022,Newport
329,1247,221,0,1135,406,130160,W06000023,Powys


In [ ]:
#santity check
print(pivot_df.columns.tolist())
pivot_df.to_sql("ethnicity_pivot", conn, if_exists="replace", index=False)


['asian_pop', 'black_pop', 'mixed_pop', 'Other', 'other_pop', 'white_pop', 'la_code', 'la_name']


331

### f) Death registrations and occurrences by local authority and health board

This is calculating the weighted deaths per LA in 2023

In [51]:
df= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/death_reg_and_ocur_by_la_ hb_2023.csv")
conn = sqlite3.connect("health_inequality.db")
df.to_sql("death_reg_la_2023", conn, index=False, if_exists="replace")


210913

In [ ]:

query = """
DROP VIEW IF EXISTS death_reg_la;

CREATE VIEW death_reg_la
AS
SELECT 
    area_name,
    area_code,
   SUM(drl.deaths) as annual_deaths,
    num_all_ages,
    (SUM(drl.deaths) * 100000.0 / epa.num_all_ages) AS deaths_per_100k
FROM 
(
    SELECT 
        `Area code` AS area_code,
        `Area name` AS area_name,
        `Deaths` AS deaths
    FROM 
        death_reg_la_2023 drl
    WHERE 
        `Geography type` = 'Local Authority') drl
INNER JOIN 
    est_pop_all epa
ON
    drl.area_code = epa.code
GROUP BY 
    drl.area_code,
    drl.area_name,
    epa.num_all_ages
"""

conn.executescript(query)  
conn.commit()

query = """
SELECT 
    *
FROM 
    death_reg_la
 """
 
result_df = pd.read_sql(query, conn)
result_df

,area_name,area_code,code,annual_deaths,num_all_ages,deaths_per_100k
0,Hartlepool,E06000001,E06000001,1043.0,95366,1093.681186
1,Middlesbrough,E06000002,E06000002,1411.0,152650,924.336718
2,Redcar and Cleveland,E06000003,E06000003,1705.0,137938,1236.062579
3,Stockton-on-Tees,E06000004,E06000004,1907.0,202415,942.123854
4,Darlington,E06000005,E06000005,1214.0,110562,1098.026447
...,...,...,...,...,...,...
309,Torfaen,W06000020,W06000020,986.0,93419,1055.459810
310,Monmouthshire,W06000021,W06000021,1010.0,94572,1067.969378
311,Newport,W06000022,W06000022,1497.0,163628,914.880094
312,Powys,W06000023,W06000023,1697.0,134439,1262.282522


### g) Travel time by GPs and Hospitals Local Authority

In [52]:


df_ttgg= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/travel_time_gps_la_2019.csv")
df_tthh= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/travel_time_hosp_la_2019.csv")

conn = sqlite3.connect("health_inequality.db")

df_ttgg.to_sql("travel_time_gps_la_2019", conn, index=False, if_exists="replace")
df_tthh.to_sql("travel_time_hosp_la_2019", conn, index=False, if_exists="replace")


361

In [ ]:

query = """
DROP VIEW IF EXISTS travel_time_gps_la;

CREATE VIEW travel_time_gps_la
AS
SELECT 
   Region AS region,
   LA_Code AS la_code,
   LA_Name AS la_name,
   GP_pop AS gp_pop,
   GPPTt AS gpptt,
   GPCyct AS gpcyct,
   GPCart AS gpcart,
   GPWalkt AS gpwalkt 
FROM 
    travel_time_gps_la_2019;
    
DROP VIEW IF EXISTS travel_time_hosp_la;

CREATE VIEW travel_time_hosp_la
AS
SELECT 
   Region AS region,
   LA_Code AS la_code,
   LA_Name AS la_name,
   Hosp_pop AS hosp_pop,
   HospPTt AS hospptt,
   HospCyct AS hospcyct,
   HospCart AS hospcart,
   HospWalkt AS hospwalkt 
FROM 
    travel_time_hosp_la_2019


"""

conn.executescript(query)  
conn.commit()

query = """
SELECT 
    *
FROM 
    travel_time_gps_la
 """
 
result_df = pd.read_sql(query, conn)
result_df

,region,la_code,la_name,gp_pop,gpptt,gpcyct,gpcart,gpwalkt
0,North East,E06000047,County Durham,"234,113",13,12,9,21
1,North East,E06000005,Darlington,"48,102",12,10,8,15
2,North East,E06000001,Hartlepool,"41,907",13,10,8,15
3,North East,E06000002,Middlesbrough,"57,147",13,10,8,15
4,North East,E06000048,Northumberland,"144,383",19,16,10,31
...,...,...,...,...,...,...,...,...
356,South West,E07000187,Mendip,"49,938",18,15,10,29
357,South West,E07000188,Sedgemoor,"53,299",16,14,9,26
358,South West,E07000189,South Somerset,"73,731",18,14,9,28
359,South West,E07000190,Taunton Deane,"51,879",17,13,9,22


### h) Vaccination Coverage of Children Age 5

Joining the different months, so the data is compiled together for the year  and finding the weighted average of children vaccinated by each product for the year.

In [ ]:


vac_jm= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/vac_cov_children_jan_to_march_2023.csv")
vac_aj= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/vac_cov_children_apr_to_jun_2023.csv")
vac_jp= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/vac_cov_children_jul_to_sept_2023.csv")
vac_od= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/vac_cov_children_oct_to_dec_2023.csv")


conn = sqlite3.connect("health_inequality.db")

vac_jm.to_sql("vac_cov_children_jan_to_march_2023", conn, index=False, if_exists="replace")
vac_aj.to_sql("vac_cov_children_apr_to_jun_2023", conn, index=False, if_exists="replace")
vac_jp.to_sql("vac_cov_children_jul_to_sept_2023", conn, index=False, if_exists="replace")
vac_od.to_sql("vac_cov_children_oct_to_dec_2023", conn, index=False, if_exists="replace")



185

In [ ]:


query = """
DROP VIEW IF EXISTS vac_cov_child_jm_2023;
DROP VIEW IF EXISTS vac_cov_child_aj_2023;
DROP VIEW IF EXISTS vac_cov_child_js_2023;
DROP VIEW IF EXISTS vac_cov_child_od_2023;

CREATE VIEW vac_cov_child_jm_2023
AS
SELECT 
   `ONS upper tier local authority code` AS ons_utla_code,
   `ODS upper tier local authority code` AS ods_utla_code,
   `Upper tier local authority name` AS utla_name,
    `5 year denominator` AS five_yr_denom,
    `5 year DTaP/IPV/Hib/HepB3%` AS five_yr_all_vac_perc,
    `5 year MMR1%` AS five_yr_mmr1_perc,
    `5 year MMR2%` AS five_yr_mmr2_perc,
    `5 year DTaPIPV%` AS five_yr_dtapipv_perc,
    `5 year Hib/MenC%` AS five_yr_hibmenc_perc
FROM 
    vac_cov_children_jan_to_march_2023;
    
CREATE VIEW vac_cov_child_aj_2023
AS
SELECT 
   `ONS UTLA code` AS ons_utla_code,
   `ODS upper tier local authority code` AS ods_utla_code,
   `UTLA name` AS utla_name,
    `5y denominator` AS five_yr_denom,
    `5y DTaP/IPV/Hib/HepB3%` AS five_yr_all_vac_perc,
    `5y MMR1%` AS five_yr_mmr1_perc,
    `5y MMR2%` AS five_yr_mmr2_perc,
    `5y DTaPIPV%` AS five_yr_dtapipv_perc,
    `5y Hib/MenC%` AS five_yr_hibmenc_perc
FROM
    vac_cov_children_apr_to_jun_2023;

CREATE VIEW vac_cov_child_js_2023
AS
SELECT 
   `ONS UTLA code` AS ons_utla_code,
   `ODS upper tier local authority code` AS ods_utla_code,
   `UTLA name` AS utla_name,
    `5y denominator` AS five_yr_denom,
    `5y DTaP/IPV/Hib/HepB3%` AS five_yr_all_vac_perc,
    `5y MMR1%` AS five_yr_mmr1_perc,
    `5y MMR2%` AS five_yr_mmr2_perc,
    `5y DTaPIPV%` AS five_yr_dtapipv_perc,
    `5y Hib/MenC%` AS five_yr_hibmenc_perc
FROM
    vac_cov_children_jul_to_sept_2023;

CREATE VIEW vac_cov_child_od_2023
AS
SELECT 
   `ONS UTLA code` AS ons_utla_code,
   `ODS upper tier local authority code` AS ods_utla_code,
   `UTLA name` AS utla_name,
    `5y denominator` AS five_yr_denom,
    `5y DTaP/IPV/Hib/HepB3%` AS five_yr_all_vac_perc,
    `5y MMR1%` AS five_yr_mmr1_perc,
    `5y MMR2%` AS five_yr_mmr2_perc,
    `5y DTaPIPV%` AS five_yr_dtapipv_perc,
    `5y Hib/MenC%` AS five_yr_hibmenc_perc
FROM
    vac_cov_children_oct_to_dec_2023;


"""

conn.executescript(query)  
conn.commit()



 ### i) Weighted Annual Vaccine 
 To find the weighting I've caluclated 

In [53]:
query = """
DROP VIEW IF EXISTS weighted_anu_vac_la_avg;

CREATE VIEW weighted_anu_vac_la_avg
AS
SELECT 
    utla_name,
  --  five_yr_denom,
  --  five_yr_all_vac_perc,
  --  five_yr_mmr1_perc,
    SUM(five_yr_denom * (five_yr_mmr1_perc /100))/SUM(five_yr_denom) AS wightd_mmrc1_perc_vac,
    SUM(five_yr_denom * (five_yr_mmr2_perc /100))/SUM(five_yr_denom) AS wightd_mmrc2_perc_vac,
    SUM(five_yr_denom * (five_yr_dtapipv_perc /100))/SUM(five_yr_denom) AS wightd_dtapipv_perc_vac,
    SUM(five_yr_denom * (five_yr_hibmenc_perc /100))/SUM(five_yr_denom) AS wightd_hibmenc_perc_vac
FROM 
(
SELECT 
    *
FROM 
    vac_cov_child_jm_2023
UNION ALL
SELECT 
    *
FROM
    vac_cov_child_aj_2023 
UNION ALL
SELECT 
    *
FROM 
    vac_cov_child_js_2023
UNION ALL
SELECT 
    *
FROM
    vac_cov_child_od_2023 
)
WHERE 
    utla_name IS NOT NULL
 GROUP BY 
    utla_name

 """
 
conn.executescript(query)  

query = """
SELECT 
    *
FROM 
    weighted_anu_vac_la_avg
 """
 
result_df = pd.read_sql(query, conn)
result_df


,utla_name,wightd_mmrc1_perc_vac,wightd_mmrc2_perc_vac,wightd_dtapipv_perc_vac,wightd_hibmenc_perc_vac
0,Barking and Dagenham,0.833014,0.687090,0.692551,0.810610
1,Barnet,0.860000,0.730250,0.723000,0.832250
2,Barnsley,0.963395,0.915911,0.906434,0.934690
3,Bath and North East Somerset,0.965039,0.928399,0.923580,0.954431
4,Bedford,0.935442,0.887343,0.878777,0.909990
...,...,...,...,...,...
148,Wirral,0.957858,0.896285,0.889393,0.952222
149,Wokingham,0.958005,0.914589,0.902553,0.941090
150,Wolverhampton,0.907475,0.787044,0.769855,0.896200
151,Worcestershire,0.959000,0.904500,0.896000,0.952500


### j) Local Authoirty Emissions Data

 calculating the emissions per person

In [ ]:

df_em= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/2005_23_uk_la_ghg_emissions.csv")

conn = sqlite3.connect("health_inequality.db")

df_em.to_sql("2005_23_uk_la_ghg_emissions", conn, index=False, if_exists="replace")
df_em

,Country,Country Code,Region,Region Code,Second Tier Authority,Local Authority,Local Authority Code,Calendar Year,LA GHG Sector,LA GHG Sub-sector,Greenhouse gas,Territorial emissions (kt CO2e),Emissions within the scope of influence of LAs (kt CO2),Mid-year Population (thousands),Area (km2)
0,England,E92000001,North East,E12000001,Hartlepool,Hartlepool,E06000001,2005,Agriculture,Agriculture Electricity,CO2,1.690511,1.690511,90.457,98.3466
1,England,E92000001,North East,E12000001,Hartlepool,Hartlepool,E06000001,2005,Agriculture,Agriculture Electricity,CH4,0.053500,0.000000,90.457,98.3466
2,England,E92000001,North East,E12000001,Hartlepool,Hartlepool,E06000001,2005,Agriculture,Agriculture Electricity,N2O,0.006820,0.000000,90.457,98.3466
3,England,E92000001,North East,E12000001,Hartlepool,Hartlepool,E06000001,2005,Agriculture,Agriculture Gas,CO2,0.264576,0.264576,90.457,98.3466
4,England,E92000001,North East,E12000001,Hartlepool,Hartlepool,E06000001,2005,Agriculture,Agriculture Gas,CH4,0.008360,0.000000,90.457,98.3466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559210,Wales,W92000004,Wales,W92000004,Wales,Merthyr Tydfil,W06000024,2023,Transport,Transport 'Other',N2O,0.007660,0.000000,58.593,111.9569
559211,Wales,W92000004,Wales,W92000004,Wales,Merthyr Tydfil,W06000024,2023,Waste,Landfill,CH4,6.562960,0.000000,58.593,111.9569
559212,Wales,W92000004,Wales,W92000004,Wales,Merthyr Tydfil,W06000024,2023,Waste,Waste 'Other',CO2,0.028900,0.028900,58.593,111.9569
559213,Wales,W92000004,Wales,W92000004,Wales,Merthyr Tydfil,W06000024,2023,Waste,Waste 'Other',CH4,2.554531,0.000000,58.593,111.9569


In [ ]:

query = """
DROP VIEW IF EXISTS la_ghg_emissions;

CREATE VIEW la_ghg_emissions
AS
SELECT 
    sta,
    la_name,
    la_code,
    ROUND((SUM(ter_emissions_co2e*1000.0)/SUM(yr_popu_thsd*1000.0)),3) AS emissions_pp,
    ROUND(SUM(ter_emissions_co2e*1000)/SUM(area_km2)) AS emissions_tons
FROM
(
SELECT 
  Country AS country,
  `Country Code` AS country_code,
  Region AS region,
  `Region Code` AS region_code,
  `Second Tier Authority` AS sta,
  `Local Authority` AS la_name,
  `Local Authority Code` AS la_code,
  `Calendar Year` AS year,
  `LA GHG Sector` AS la_ghg_sector,
  `LA GHG Sub-sector` AS la_ghg_sub_sec,
  `Greenhouse gas` AS greenhouse_gas,
  `Territorial emissions (kt CO2e)` AS ter_emissions_co2e,
  `Emissions within the scope of influence of LAs (kt CO2)` AS em_sc_las_co2,
  `Mid-year Population (thousands)` AS yr_popu_thsd,
  `Area (km2)` AS area_km2
FROM 
    `2005_23_uk_la_ghg_emissions`
WHERE 
    year = 2023 )
GROUP BY 
    sta
"""

conn.executescript(query)  
conn.commit()

query = """
SELECT 
    *
FROM 
    la_ghg_emissions
 """
 
result_df = pd.read_sql(query, conn)
result_df

## QOF Data

The quality of outcomes data is collected from GP practices looking at the quality of care they provide to their patients, based on several indicators across a range of key areas of clinical care and public health. 

Within this project, it is looking at:
- Atrial fibrillation
- Coronary Heart Disease 
 -Blood Preassure
- Heart Failure
- Hypertension
- Asthma
- COPD
- Obesity
- Smoking
- Demand and Staff Capcity
- Reducing avoidable appointments

In [56]:
df_af= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_af.csv")
df_chd= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_chd.csv")
df_bp = pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_bp.csv")
df_hf= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_hf.csv")
df_hyp= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_hyp.csv")
df_bp = pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_bp.csv")
df_ast= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_ast.csv")
df_copd = pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_copd.csv")
df_ob= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_ob.csv")
df_smo = pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_pca_gp_smo.csv")

df_qiraa= pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_qual_imp_prac_qiraa.csv")
df_qiosc = pd.read_csv("/Users/lulum/Documents/Coding Projects/01 PRJ  - UK Health Inequality/datasets/qof_2324_prev_ach_qual_imp_prac_qiosc.csv")

conn = sqlite3.connect("health_inequality.db")

df_af.to_sql("qof_2324_prev_ach_pca_gp_af", conn, index=False, if_exists="replace")
df_chd.to_sql("qof_2324_prev_ach_pca_gp_chd", conn, index=False, if_exists="replace")
df_bp.to_sql("qof_2324_prev_ach_pca_gp_bp", conn, index=False, if_exists="replace")
df_hf.to_sql("qof_2324_prev_ach_pca_gp_hf", conn, index=False, if_exists="replace")
df_hyp.to_sql("qof_2324_prev_ach_pca_gp_hyp", conn, index=False, if_exists="replace")
df_chd.to_sql("qof_2324_prev_ach_pca_gp_chd", conn, index=False, if_exists="replace")
df_ast.to_sql("qof_2324_prev_ach_pca_gp_ast", conn, index=False, if_exists="replace")
df_copd.to_sql("qof_2324_prev_ach_pca_gp_copd", conn, index=False, if_exists="replace")
df_ob.to_sql("qof_2324_prev_ach_pca_gp_ob", conn, index=False, if_exists="replace")
df_smo.to_sql("qof_2324_prev_ach_pca_gp_smo", conn, index=False, if_exists="replace")
df_qiraa.to_sql("qof_2324_prev_ach_qual_imp_prac_qiraa", conn, index=False, if_exists="replace")
df_qiosc.to_sql("qof_2324_prev_ach_qual_imp_prac_qiosc", conn, index=False, if_exists="replace")



7322

In [ ]:

query = """
DROP VIEW IF EXISTS qof_gp_bp;

CREATE VIEW qof_gp_bp
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_bp) AS total_list_size_bp,
    SUM(qf.ttl_achiev_score) AS total_reg_bp,
    (SUM(qf.ttl_achiev_score) * 1.0 / SUM(qf.list_size_bp) * 100) AS weighted_prev_bp
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size \naged 45+`, ',', '') AS INTEGER) AS list_size_bp,
        CAST(REPLACE(`Total Achievement Score (max 15)`, ',', '') AS INTEGER) AS ttl_achiev_score,
        `Achievement (%)` AS prevalence_bp
    FROM 
     `qof_2324_prev_ach_pca_gp_bp`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
        ON qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
"""


conn.executescript(query)  
conn.commit()

In [ ]:

query = """
DROP VIEW IF EXISTS qof_gp_af;

CREATE VIEW qof_gp_af
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qaf.list_size_af) AS total_list_size_af,
    SUM(qaf.register_af) AS total_reg_af,
    (SUM(qaf.register_af) * 1.0 / SUM(qaf.list_size_af) * 100) AS weighted_prev_af
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size`, ',', '') AS INTEGER) AS list_size_af,
        CAST(REPLACE(`Register`, ',', '') AS INTEGER) AS register_af,
        `Prevalence (%)` AS prevalence_af
    FROM 
     `qof_2324_prev_ach_pca_gp_af`
        ) qaf
LEFT JOIN 
    epraccur_lookup epl
        ON qaf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
    

DROP VIEW IF EXISTS qof_gp_chd;

CREATE VIEW qof_gp_chd
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_chd) AS total_list_size_chd,
    SUM(qf.register_chd) AS total_reg_chd,
    (SUM(qf.register_chd) * 1.0 / SUM(qf.list_size_chd) * 100) AS weighted_prev_chd
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size`, ',', '') AS INTEGER) AS list_size_chd,
        CAST(REPLACE(`Register`, ',', '') AS INTEGER) AS register_chd,
        `Prevalence (%)` AS prevalence_chd
    FROM 
     `qof_2324_prev_ach_pca_gp_chd`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
    ON 
    qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
    ON 
    REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
"""

conn.executescript(query)  

query = """
SELECT
*
FROM
    qof_gp_chd
"""
result_df = pd.read_sql(query, conn)
result_df

,ladnm,ladcd,total_list_size_chd,total_reg_chd,weighted_prev_chd
0,None,None,NaN,NaN,NaN
1,Hartlepool,E06000001,98927.0,4063.0,4.107069
2,Middlesbrough,E06000002,172190.0,5473.0,3.178466
3,Redcar and Cleveland,E06000003,138953.0,5542.0,3.988399
4,Stockton-on-Tees,E06000004,209702.0,7616.0,3.631820
...,...,...,...,...,...
292,Sutton,E09000029,216240.0,5056.0,2.338143
293,Tower Hamlets,E09000030,388446.0,5677.0,1.461464
294,Waltham Forest,E09000031,329133.0,5864.0,1.781651
295,Wandsworth,E09000032,411866.0,5717.0,1.388073


In [ ]:

query = """
DROP VIEW IF EXISTS qof_gp_hf;

CREATE VIEW qof_gp_hf
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_hf) AS total_list_size_hf,
    SUM(qf.register_hf) AS total_reg_hf,
    (SUM(qf.register_hf) * 1.0 / SUM(qf.list_size_hf) * 100) AS weighted_prev_hf
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size`, ',', '') AS INTEGER) AS list_size_hf,
        CAST(REPLACE(`Register`, ',', '') AS INTEGER) AS register_hf,
        `Prevalence (%)` AS prevalence_hf
    FROM 
     `qof_2324_prev_ach_pca_gp_hf`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
        ON qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
    
DROP VIEW IF EXISTS qof_gp_hyp;

CREATE VIEW qof_gp_hyp
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_hyp) AS total_list_size_hyp,
    SUM(qf.register_hyp) AS total_reg_hyp,
    (SUM(qf.register_hyp) * 1.0 / SUM(qf.list_size_hyp) * 100) AS weighted_pre_hyp
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size`, ',', '') AS INTEGER) AS list_size_hyp,
        CAST(REPLACE(`Register`, ',', '') AS INTEGER) AS register_hyp,
        `Prevalence (%)` AS prevalence_hyp
    FROM 
     `qof_2324_prev_ach_pca_gp_hyp`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
        ON qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
"""

conn.executescript(query)  

query = """
SELECT
*
FROM
    qof_gp_hf
"""
result_df = pd.read_sql(query, conn)
result_df

In [ ]:

query = """
DROP VIEW IF EXISTS qof_gp_chd;

CREATE VIEW qof_gp_chd
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_chd) AS total_list_size_chd,
    SUM(qf.register_chd) AS total_reg_chd,
    (SUM(qf.register_chd) * 1.0 / SUM(qf.list_size_chd) * 100) AS weighted_prev_chd
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size`, ',', '') AS INTEGER) AS list_size_chd,
        CAST(REPLACE(`Register`, ',', '') AS INTEGER) AS register_chd,
        `Prevalence (%)` AS prevalence_chd
    FROM 
     `qof_2324_prev_ach_pca_gp_chd`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
        ON qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
    
DROP VIEW IF EXISTS qof_gp_ast;

CREATE VIEW qof_gp_ast
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_ast) AS total_list_size_ast,
    SUM(qf.register_ast) AS total_register_ast,
    (SUM(qf.register_ast) * 1.0 / SUM(qf.list_size_ast) * 100) AS weighted_prev_ast
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size 
aged 6+`, ',', '') AS INTEGER) AS list_size_ast,
        CAST(REPLACE(`Register`, ',', '') AS INTEGER) AS register_ast,
        `Prevalence (%)` AS prevalence_ast
    FROM 
     `qof_2324_prev_ach_pca_gp_ast`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
        ON qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
    
DROP VIEW IF EXISTS qof_gp_copd;

CREATE VIEW qof_gp_copd
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_copd) AS total_list_size_copd,
    SUM(qf.register_copd) AS total_reg_copd,
    (SUM(qf.register_copd) * 1.0 / SUM(qf.list_size_copd) * 100) AS weighted_prev_copd
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size`, ',', '') AS INTEGER) AS list_size_copd,
        CAST(REPLACE(`Register`, ',', '') AS INTEGER) AS register_copd,
        `Prevalence (%)` AS prevalence_copd
    FROM 
     `qof_2324_prev_ach_pca_gp_copd`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
        ON qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
"""

conn.executescript(query)  
conn.commit()

query = """
SELECT
*
FROM
    qof_gp_chd
"""
result_df = pd.read_sql(query, conn)
result_df

In [ ]:

query = """
DROP VIEW IF EXISTS qof_gp_smo;

CREATE VIEW qof_gp_smo
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_smo) AS total_list_size_smo,
    (SUM(qf.ttl_achiev_score) * 1.0 / SUM(qf.list_size_smo) * 100) AS weighted_achiev_smo
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size \naged 15+`, ',', '') AS INTEGER) AS list_size_smo,
        CAST(REPLACE(`Total Achievement Score (max 62)`, ',', '') AS INTEGER) AS ttl_achiev_score,
        `Achievement (%)` AS achev_smo
    FROM 
     `qof_2324_prev_ach_pca_gp_smo`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
        ON qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
    
DROP VIEW IF EXISTS qof_gp_obo;

CREATE VIEW qof_gp_obo
AS

SELECT
    pc.ladnm,
    pc.ladcd,
    SUM(qf.list_size_obo) AS total_list_size_obo,
    SUM(qf.register_obo) AS total_reg_obo,
    (SUM(qf.register_obo) * 1.0 / SUM(qf.list_size_obo) * 100) AS weighted_prev_obo
FROM
    (
    SELECT 
        `Sub ICB Loc ODS code` AS sicb_loc_ods_code,
        `Sub ICB Loc ONS code` AS sicb_loc_ons_code,
        `Sub ICB Loc name` AS sicb_loc_name,
        `PCN ODS code` AS pcn_ods_code,
        `PCN name` AS pcn_name,
        `Practice code` AS practice_code,
        `Practice name` AS practice_name,
        CAST(REPLACE(`List size 
aged 18+`, ',', '') AS INTEGER) AS list_size_obo,
        CAST(REPLACE(`Register`, ',', '') AS INTEGER) AS register_obo,
        `Prevalence (%)` AS prevalence_obo
    FROM 
     `qof_2324_prev_ach_pca_gp_ob`
        ) qf
LEFT JOIN 
    epraccur_lookup epl
        ON qf.practice_code = epl.org_code
LEFT JOIN 
    clean_postcode pc
        ON REPLACE(UPPER(epl.postcode), ' ', '') = pc.clean_pcds
GROUP BY
    pc.ladcd,
    pc.ladnm;
"""

conn.executescript(query)  

query = """
SELECT
*
FROM
    qof_gp_obo
"""
result_df = pd.read_sql(query, conn)
result_df

,ladnm,ladcd,total_list_size_obo,total_reg_obo,weighted_prev_obo
0,None,None,NaN,NaN,NaN
1,Hartlepool,E06000001,78718.0,14485.0,18.401128
2,Middlesbrough,E06000002,134391.0,20094.0,14.951894
3,Redcar and Cleveland,E06000003,112037.0,18437.0,16.456171
4,Stockton-on-Tees,E06000004,166072.0,25254.0,15.206657
...,...,...,...,...,...
292,Sutton,E09000029,168261.0,19978.0,11.873221
293,Tower Hamlets,E09000030,319582.0,30265.0,9.470183
294,Waltham Forest,E09000031,259228.0,32617.0,12.582360
295,Wandsworth,E09000032,343503.0,27052.0,7.875332



## Final Master Analytical Dataset - Local Authority

This is the final dataset joining together all cleaned and aggregated LA views into a single unified analytical table. All joins have been performed at a LA level to ensure granularity and prevent duplication.

Each dataset was previously:
- cleaned
- standardied
- aggegated
- validated

*The final Dataset includes*: 
- Demographics (Population, Ethnicity)
- Socioeconomic Indicators (IMD, Education, Employment, Housing)
- Health Outcomes (Life Expectancy, Mortality Rate)
- Primary Care Access (GP Patient Volume, Travel Time)
- Clinical Indicators (QOF disease prevalence and achievement)
- Environmental Indicators (GHG emissions)
- Vaccination Coverage



In [ ]:


query = """
DROP TABLE IF EXISTS clean_postcode;

CREATE TABLE clean_postcode AS
SELECT DISTINCT
    REPLACE(UPPER(pcds), ' ', '') AS clean_pcds,
    ladnm,
    ladcd
FROM postcode_nov_2025_lookup;

CREATE INDEX idx_clean_postcode ON clean_postcode(clean_pcds);
"""

conn.executescript(query)

pd.read_sql("SELECT * FROM clean_postcode LIMIT 20", conn)



,clean_pcds,ladnm,ladcd
0,AB10AA,Aberdeen City,S12000033
1,AB10AB,Aberdeen City,S12000033
2,AB10AD,Aberdeen City,S12000033
3,AB10AE,Aberdeenshire,S12000034
4,AB10AF,Aberdeen City,S12000033
5,AB10AG,Aberdeen City,S12000033
6,AB10AJ,Aberdeen City,S12000033
7,AB10AL,Aberdeen City,S12000033
8,AB10AN,Aberdeen City,S12000033
9,AB10AP,Aberdeen City,S12000033


In [ ]:

query = """
DROP VIEW IF EXISTS health_inequality_m;

CREATE VIEW health_inequality_m AS
SELECT
    --Life Expectancy
    --life.code,
    life.area_code,
    life.area_name,
    life.weighted_le AS 'Life expectancy',
    
     --Population
    --pop.code,
    --pop.name,
    --Geography AS geography,
     pop.num_all_ages AS Population,
    
   -- gp.ladnm AS gp_lad_name,
    --gp.ladcd AS gp_lad_code,
    gp.sum_patients_reg_la AS 'Amount of GP Patients',
    
     -- Deprivation
   -- dep.la_name AS dep_la_name,
   -- dep.la_code AS dep_la_code,
    dep.imd_rank_avg AS 'Avg IMD Rank',
    dep.imd_decile_avg AS 'Avg IMD Decile', 
    dep.edu_rank_avg AS 'Avg Education Rank', 
    dep.health_rank_avg AS 'Avg Health Rank', 
    dep.barriers_rank_avg AS 'Avg Barriers Rank', 
    dep.living_env_rank_avg AS 'Avg Living Environment Rank', 
    dep.employment_rank_avg AS 'AVG Employment Rank',

   -- eth.la_name AS eth_la_name,
    --eth.la_code AS eth_la_code,
    eth.white_pop AS 'White Popua',
    eth.asian_pop,
    eth.black_pop,
    eth.mixed_pop,
    eth.other_pop,
    
        -- Deaths
    --death.area_name AS death_la_name,
    --death.area_code AS death_la_code,
    death.deaths_per_100k as 'Deaths per 100k',
    
    -- GP Travel Time
    --gp_tt.la_name AS gp_tt_la_name,
    gp_tt.gpptt AS 'GP time walking' ,
    gp_tt.gpcyct,
    gp_tt.gpcart,
    gp_tt.gpwalkt,

    -- Hospital Travel Time
    --hosp_tt.la_name AS hosp_tt_la_name,
    hosp_tt.hospptt,
    hosp_tt.hospcyct,
    hosp_tt.hospcart,
    hosp_tt.hospwalkt,
    
     -- Vaccination Coverage
   -- vac.utla_name AS vac_la_name,
    vac.wightd_mmrc1_perc_vac,
    vac.wightd_mmrc2_perc_vac,
    vac.wightd_dtapipv_perc_vac,
    vac.wightd_hibmenc_perc_vac,

    -- GHG Emissions
    --ghg.la_name AS ghg_la_name,
    --ghg.la_code AS ghg_la_code,
    --ghg.sta,
    ghg.emissions_pp,
    ghg.emissions_tons,
    
     -- QOF: Atrial Fibrillation
    --af.ladnm AS af_la_name,
    --af.ladcd AS af_la_code,
    af.total_list_size_af AS af_list_size,
    af.total_reg_af AS af_register,
    af.weighted_prev_af AS af_prevalence,

    -- QOF: Coronary Heart Disease
    chd.ladnm AS chd_la_name,
    chd.ladcd AS chd_la_code,
    chd.total_list_size_chd AS chd_list_size,
    chd.total_reg_chd AS chd_register,
    chd.weighted_prev_chd AS chd_prevalence,

    -- QOF: Blood Preassure
    bp.total_list_size_bp,
    bp.total_reg_bp,
    bp.weighted_prev_bp,
    
    -- QOF: Heart Failure
    --hf.ladnm AS hf_la_name,
    --hf.ladcd AS hf_la_code,
    hf.total_list_size_hf,
    hf.total_reg_hf,
    hf.weighted_prev_hf,

    -- QOF: Hypertension
    --hyp.ladnm AS hyp_la_name,
    --hyp.ladcd AS hyp_la_code,
    hyp.total_list_size_hyp,
    hyp.total_reg_hyp,
    hyp.weighted_pre_hyp,
    
        -- QOF: COPD
    --copd.ladnm AS copd_la_name,
    --copd.ladcd AS copd_la_code,
    copd.total_list_size_copd,
    copd.total_reg_copd,
    copd.weighted_prev_copd,
    
        -- QOF: Asthma
    --ast.ladnm AS ast_la_name,
    --ast.ladcd AS ast_la_code,
    ast.total_list_size_ast,
    ast.total_register_ast,
    ast.weighted_prev_ast,

    -- QOF: Smoking
   -- smo.ladnm AS smo_la_name,
   -- smo.ladcd AS smo_la_code,
    smo.total_list_size_smo,
    smo.weighted_achiev_smo,

    -- QOF: Obesity
    --obo.ladnm AS obo_la_name,
    --obo.ladcd AS obo_la_code,
    obo.total_list_size_obo,
    obo.total_reg_obo,
    obo.weighted_prev_obo
    
        -- QOF: optimise use of staff capacity
   -- smo.ladnm AS smo_la_name,
   -- smo.ladcd AS smo_la_code,
   -- qiosc.total_list_size_qiosc,
   -- qiosc.weighted_achiev_qiosc,
    
    -- QOF: Reducing avoidable appointments
   -- smo.ladnm AS smo_la_name,
   -- smo.ladcd AS smo_la_code,
   -- qiraa.total_list_size_qiraa,
    --qiraa.weighted_achiev_qiraa,
    
FROM life_expect life

LEFT JOIN est_pop_all pop
       ON pop.code = life.area_code 

LEFT JOIN gp_reg_pat_prac gp
       ON gp.ladcd = life.area_code 
       
LEFT JOIN deprivation_la dep
       ON dep.la_code = life.area_code

LEFT JOIN ethnicity_pivot eth
       ON eth.la_code = life.area_code
    
LEFT JOIN death_reg_la death
       ON death.area_code = life.area_code 
     
LEFT JOIN travel_time_gps_la gp_tt
       ON gp_tt.la_code = life.area_code 
     
LEFT JOIN travel_time_hosp_la hosp_tt
       ON hosp_tt.la_code = life.area_code 
        
LEFT JOIN weighted_anu_vac_la_avg vac
       ON vac.utla_name = life.area_name

LEFT JOIN la_ghg_emissions ghg
       ON ghg.la_code = life.area_code 
     
LEFT JOIN qof_gp_af af
       ON af.ladcd = life.area_code 
     
LEFT JOIN qof_gp_chd chd
       ON chd.ladcd = life.area_code 
     
LEFT JOIN qof_gp_bp bp
       ON bp.ladcd = life.area_code 
     
LEFT JOIN qof_gp_hf hf
       ON hf.ladcd = life.area_code 
      
LEFT JOIN qof_gp_hyp hyp
       ON hyp.ladcd = life.area_code 
      
LEFT JOIN qof_gp_copd copd
       ON copd.ladcd = life.area_code 
      
LEFT JOIN qof_gp_ast ast
       ON ast.ladcd = life.area_code 
       
LEFT JOIN qof_gp_smo smo
       ON smo.ladcd = life.code 
       
LEFT JOIN qof_gp_obo obo
       ON obo.ladcd = life.area_code 
      
--LEFT JOIN qof_gp_qiraa qiraa
     --  ON qiraa.ladcd = qiraa.area_code 
    --   OR qiraa.ladnm = qira.area_name  
       
--LEFT JOIN qof_gp_qiosc qiosc
    --   ON qiosc.ladcd = qiosc.area_code 
    --   OR qiosc.ladnm = qiosc.area_name  
       """

conn.executescript(query)

result_df = pd.read_sql("SELECT * FROM health_inequality_m", conn)
result_df.head()
result_df.to_csv("health_inequality_dataset.csv", index=False)


with pd.option_context('display.max_rows', None):
    display(result_df)


In [61]:
print(result_df.shape)
print(result_df.isnull().sum().sort_values(ascending=False).head())


(294, 60)
wightd_hibmenc_perc_vac    171
wightd_dtapipv_perc_vac    171
wightd_mmrc2_perc_vac      171
wightd_mmrc1_perc_vac      171
emissions_pp               148
dtype: int64
